# 生日五行金融密碼計算器

使用下方 widget 選擇西元出生年月日，按下計算按鈕後，系統會計算生日數字密碼、對應河圖五行，並以 Plotly 顯示互動式五行圖與金融投資風格建議。

> 本內容僅供課堂學習與投資風格分析，不構成投資建議。

In [ ]:
from datetime import date
import math

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import HTML, display

In [ ]:
WUXING_BY_DIGIT = {
    1: "木",
    6: "木",
    2: "火",
    7: "火",
    5: "土",
    0: "土",
    4: "金",
    9: "金",
    3: "水",
    8: "水",
}

WUXING_COLORS = {
    "木": "#2E8B57",
    "火": "#E4572E",
    "土": "#B08D57",
    "金": "#C9A227",
    "水": "#2F80ED",
}

WUXING_ADVICE = {
    "木": {
        "summary": "成長型、重視長期累積與趨勢延伸。",
        "advice": "木屬性適合建立長期投資紀律，可關注具成長性、研發能力或長線需求支撐的產業。操作上宜避免只追短線題材，應重視基本面、定期檢視與分散配置。",
    },
    "火": {
        "summary": "行動型、反應快，容易受市場情緒帶動。",
        "advice": "火屬性適合明確設定進出場規則與風險上限，可研究動能、景氣循環與市場熱度變化。需特別避免過度交易與情緒化加碼，建議用停損、停利與部位控管降低波動風險。",
    },
    "土": {
        "summary": "穩健型、重視安全感與資產配置平衡。",
        "advice": "土屬性適合以資產配置為核心，重視現金流、風險分散與投資組合穩定度。可採取定期定額或核心衛星配置，避免因過度保守而完全錯失長期成長機會。",
    },
    "金": {
        "summary": "紀律型、重視規則、效率與風險報酬比。",
        "advice": "金屬性適合使用量化條件、財務指標與明確篩選規則做投資決策。可重視價值評估、財務體質與風險報酬比，但需避免過度僵化，仍要留意市場環境變化。",
    },
    "水": {
        "summary": "觀察型、擅長資訊整合與等待機會。",
        "advice": "水屬性適合重視研究、資料蒐集與情境推演，可觀察利率、匯率、資金流與總體經濟變化。需避免因資訊過多而猶豫不決，建議事先設定可執行的投資條件。",
    },
}


def split_digits(number):
    """將整數拆成各位數字清單。

    Args:
        number: 要拆解的非負整數。

    Returns:
        list[int]: 由每一位數字組成的清單。
    """
    return [int(digit) for digit in str(number)]


def calculate_birthday_password(birthday):
    """計算生日數字密碼並保留每次拆位加總過程。

    Args:
        birthday: 使用者選擇的西元生日日期。

    Returns:
        tuple[int, list[tuple[list[int], int]]]: 最終 0 到 9 的生日密碼，
        以及每次計算使用的數字清單與加總結果。
    """
    birthday_digits = [int(digit) for digit in birthday.strftime("%Y%m%d")]
    steps = []
    current_digits = birthday_digits

    while True:
        total = sum(current_digits)
        steps.append((current_digits, total))
        if total < 10:
            return total, steps
        current_digits = split_digits(total)


def format_steps(steps):
    """將生日密碼計算步驟格式化成可在 Notebook 顯示的 HTML 文字。

    Args:
        steps: calculate_birthday_password 回傳的計算步驟。

    Returns:
        str: 使用 <br> 分隔的加總流程文字。
    """
    lines = []
    for index, (digits, total) in enumerate(steps, start=1):
        expression = " + ".join(str(digit) for digit in digits)
        lines.append(f"第 {index} 次：{expression} = {total}")
    return "<br>".join(lines)

In [ ]:
def create_wuxing_figure(selected_digit=None):
    """建立 Plotly 河圖五行數字對應圖。

    Args:
        selected_digit: 要高亮顯示的生日密碼數字；未提供時只顯示完整對應圖。

    Returns:
        plotly.graph_objects.Figure: 可在 Notebook 中互動檢視的五行圖表。
    """
    digits = list(range(10))
    radius = 2.3
    digit_points = []

    for index, digit in enumerate(digits):
        angle = math.pi / 2 - (2 * math.pi * index / len(digits))
        element = WUXING_BY_DIGIT[digit]
        digit_points.append(
            {
                "digit": digit,
                "element": element,
                "x": radius * math.cos(angle),
                "y": radius * math.sin(angle),
                "color": WUXING_COLORS[element],
                "summary": WUXING_ADVICE[element]["summary"],
            }
        )

    element_order = ["木", "火", "土", "金", "水"]
    element_points = []
    for index, element in enumerate(element_order):
        angle = math.pi / 2 - (2 * math.pi * index / len(element_order))
        element_points.append(
            {
                "element": element,
                "x": 0.85 * math.cos(angle),
                "y": 0.85 * math.sin(angle),
                "color": WUXING_COLORS[element],
                "digits": ", ".join(str(d) for d in digits if WUXING_BY_DIGIT[d] == element),
            }
        )

    fig = go.Figure()

    for point in digit_points:
        element_point = next(item for item in element_points if item["element"] == point["element"])
        fig.add_trace(
            go.Scatter(
                x=[element_point["x"], point["x"]],
                y=[element_point["y"], point["y"]],
                mode="lines",
                line={"color": point["color"], "width": 1.5},
                hoverinfo="skip",
                showlegend=False,
            )
        )

    fig.add_trace(
        go.Scatter(
            x=[point["x"] for point in element_points],
            y=[point["y"] for point in element_points],
            mode="markers+text",
            text=[point["element"] for point in element_points],
            textposition="middle center",
            marker={
                "size": 58,
                "color": [point["color"] for point in element_points],
                "line": {"color": "#333333", "width": 1},
            },
            textfont={"size": 20, "color": "white"},
            customdata=[[point["digits"]] for point in element_points],
            hovertemplate="五行：%{text}<br>對應數字：%{customdata[0]}<extra></extra>",
            name="五行",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[point["x"] for point in digit_points],
            y=[point["y"] for point in digit_points],
            mode="markers+text",
            text=[str(point["digit"]) for point in digit_points],
            textposition="middle center",
            marker={
                "size": [64 if point["digit"] == selected_digit else 42 for point in digit_points],
                "color": [point["color"] for point in digit_points],
                "line": {
                    "color": ["#111111" if point["digit"] == selected_digit else "#FFFFFF" for point in digit_points],
                    "width": [5 if point["digit"] == selected_digit else 2 for point in digit_points],
                },
            },
            textfont={"size": 18, "color": "white"},
            customdata=[[point["element"], point["summary"]] for point in digit_points],
            hovertemplate="數字：%{text}<br>五行：%{customdata[0]}<br>%{customdata[1]}<extra></extra>",
            name="生日密碼數字",
        )
    )

    title = "河圖五行數字對應圖"
    if selected_digit is not None:
        title += f"（已高亮：{selected_digit}）"

    fig.update_layout(
        title={"text": title, "x": 0.5},
        width=760,
        height=620,
        margin={"l": 20, "r": 20, "t": 80, "b": 20},
        paper_bgcolor="#FAFAFA",
        plot_bgcolor="#FAFAFA",
        showlegend=False,
        xaxis={"visible": False, "range": [-3, 3]},
        yaxis={"visible": False, "range": [-3, 3], "scaleanchor": "x", "scaleratio": 1},
    )
    return fig

In [ ]:
birthday_picker = widgets.DatePicker(
    description="出生日期",
    disabled=False,
    value=date(2000, 1, 1),
)

calculate_button = widgets.Button(
    description="計算生日五行金融密碼",
    button_style="primary",
    icon="calculator",
    layout=widgets.Layout(width="240px"),
)

result_output = widgets.Output()


def render_result(_button):
    """處理計算按鈕點擊事件並更新 Notebook 輸出區。

    Args:
        _button: ipywidgets 傳入的按鈕事件物件；此函式不直接使用該值。

    Side Effects:
        清除舊輸出，驗證生日輸入，顯示生日密碼、五行建議與 Plotly 圖表。
    """
    with result_output:
        result_output.clear_output()

        birthday = birthday_picker.value
        if birthday is None:
            display(HTML("<p style='color:#B00020; font-weight:600;'>請先選擇出生日期。</p>"))
            return

        password, steps = calculate_birthday_password(birthday)
        element = WUXING_BY_DIGIT[password]
        advice = WUXING_ADVICE[element]

        display(
            HTML(
                f"""
                <div style="font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; line-height: 1.65;">
                    <h2 style="margin-bottom: 0.4rem;">計算結果</h2>
                    <p><strong>出生日期：</strong>{birthday.strftime('%Y-%m-%d')}</p>
                    <p><strong>拆位加總流程：</strong><br>{format_steps(steps)}</p>
                    <p><strong>生日數字密碼：</strong><span style="font-size: 1.4rem; font-weight: 700; color: #111;"> {password}</span></p>
                    <p><strong>河圖五行屬性：</strong><span style="font-size: 1.2rem; font-weight: 700; color: {WUXING_COLORS[element]};"> {element}</span></p>
                    <p><strong>投資風格摘要：</strong>{advice['summary']}</p>
                    <p><strong>金融投資建議：</strong>{advice['advice']}</p>
                    <p style="color:#6B7280;"><strong>風險提醒：</strong>本內容僅供課堂學習與投資風格分析，不構成投資建議、買賣推薦或收益保證。</p>
                </div>
                """
            )
        )
        display(create_wuxing_figure(password))


calculate_button.on_click(render_result)

display(widgets.VBox([widgets.HBox([birthday_picker, calculate_button]), result_output]))